# Unsupervised Results Analysis

Analyze `data/results/unsupervised.csv` from the unsupervised runner. The notebook ranks anomaly detection experiments, compares model configs, and inspects threshold behavior through FPR/FNR/TPR/TNR.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_PATH = PROJECT_ROOT / "data" / "results" / "unsupervised.csv"
RESULTS_PATH

In [ ]:
df = pd.read_csv(RESULTS_PATH)

numeric_cols = [
    "rows", "accuracy", "f1", "recall", "precision", "fpr", "fnr",
    "tpr", "tnr", "auc", "fit_seconds", "accuracy_std", "f1_std",
    "recall_std", "precision_std", "fpr_std", "fnr_std", "tpr_std",
    "tnr_std", "auc_std", "fit_seconds_std",
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["fold"] = df["fold"].fillna("")
df["is_fold"] = df["evaluation"].eq("fold_val")
df["is_summary"] = df["evaluation"].eq("fold_mean")
df["is_holdout"] = df["evaluation"].isin(["val", "test"])

print(df.shape)
df.head()

## Overview

In [ ]:
overview = (
    df.groupby(["model", "model_config", "evaluation"], dropna=False)
      .agg(
          runs=("experiment_name", "count"),
          mean_f1=("f1", "mean"),
          mean_auc=("auc", "mean"),
          mean_tpr=("tpr", "mean"),
          mean_fpr=("fpr", "mean"),
          mean_fit_seconds=("fit_seconds", "mean"),
      )
      .reset_index()
      .sort_values(["evaluation", "mean_f1"], ascending=[True, False])
)
overview

## Holdout Ranking

In [ ]:
holdout = df[df["is_holdout"]].copy()

rank_cols = [
    "experiment_name", "evaluation", "model", "model_config", "rows",
    "f1", "auc", "precision", "recall", "fpr", "fnr", "tpr", "tnr",
    "fit_seconds", "label_counts", "prediction_counts",
]
holdout_ranked = holdout.sort_values(["evaluation", "f1", "auc"], ascending=[True, False, False])
holdout_ranked[rank_cols].head(20)

In [ ]:
test_ranked = holdout[holdout["evaluation"].eq("test")].sort_values(["f1", "auc"], ascending=False)
test_ranked[rank_cols].head(15)

## Metric Tradeoffs

In [ ]:
plot_df = holdout[holdout["evaluation"].eq("test")].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=plot_df,
    x="fpr",
    y="tpr",
    hue="model",
    style="model_config",
    s=90,
    ax=axes[0],
)
axes[0].set_title("Test TPR vs FPR")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")

sns.scatterplot(
    data=plot_df,
    x="precision",
    y="recall",
    hue="model",
    style="model_config",
    s=90,
    ax=axes[1],
)
axes[1].set_title("Test Precision vs Recall")
axes[1].set_xlabel("Precision")
axes[1].set_ylabel("Recall")

for ax in axes:
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()

In [ ]:
metric_long = plot_df.melt(
    id_vars=["model", "model_config", "model_experiment"],
    value_vars=["f1", "auc", "precision", "recall", "fpr", "fnr", "tpr", "tnr"],
    var_name="metric",
    value_name="value",
)

plt.figure(figsize=(14, 6))
sns.barplot(data=metric_long, x="model_experiment", y="value", hue="metric")
plt.xticks(rotation=45, ha="right")
plt.title("Test Metrics by Unsupervised Experiment")
plt.xlabel("Experiment")
plt.ylabel("Metric value")
plt.tight_layout()

## Fold Stability

In [ ]:
fold_summary = df[df["is_summary"]].copy()
fold_cols = [
    "model", "model_config", "model_experiment", "rows", "f1", "f1_std",
    "auc", "auc_std", "precision", "precision_std", "recall", "recall_std",
    "fpr", "fpr_std", "fnr", "fnr_std", "fit_seconds", "fit_seconds_std",
]
fold_summary[fold_cols].sort_values(["f1", "auc"], ascending=False)

In [ ]:
folds = df[df["is_fold"]].copy()

plt.figure(figsize=(12, 5))
sns.boxplot(data=folds, x="model_experiment", y="f1")
sns.stripplot(data=folds, x="model_experiment", y="f1", color="black", alpha=0.5, size=4)
plt.xticks(rotation=45, ha="right")
plt.title("Fold Validation F1 Distribution")
plt.xlabel("Experiment")
plt.ylabel("F1")
plt.tight_layout()

## Prediction Volume

In [ ]:
def parse_counts(value):
    if pd.isna(value):
        return {}
    if isinstance(value, dict):
        return value
    return json.loads(value)

pred_counts = holdout.copy()
pred_counts["predicted_normal"] = pred_counts["prediction_counts"].apply(lambda x: parse_counts(x).get("0", 0))
pred_counts["predicted_anomaly"] = pred_counts["prediction_counts"].apply(lambda x: parse_counts(x).get("1", 0))
pred_counts["predicted_anomaly_rate"] = pred_counts["predicted_anomaly"] / pred_counts["rows"]

pred_counts[[
    "experiment_name", "evaluation", "model", "model_config", "rows",
    "predicted_anomaly", "predicted_anomaly_rate", "fpr", "tpr", "precision", "recall", "f1",
]].sort_values(["evaluation", "predicted_anomaly_rate"], ascending=[True, False])

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(
    data=pred_counts[pred_counts["evaluation"].eq("test")],
    x="model_experiment",
    y="predicted_anomaly_rate",
    hue="model",
    dodge=False,
)
plt.xticks(rotation=45, ha="right")
plt.title("Test Predicted Anomaly Rate")
plt.xlabel("Experiment")
plt.ylabel("Predicted anomaly rate")
plt.tight_layout()

## Summary

The strongest unsupervised result is `sgd_one_class_svm:loose`. On the test split it reaches F1 `0.3339`, AUC `0.5699`, precision `0.7435`, and recall `0.2153`. This means the model is conservative: when it flags a message as anomalous/toxic it is often correct, but it misses most toxic examples.

Across the unsupervised experiments, false negative rates are high. The best test model still has FNR `0.7847`, so roughly four out of five toxic examples are not flagged. This makes the current unsupervised setup weak as a standalone toxicity detector if recall matters.

`SGDOneClassSVM` is consistently better than `IsolationForest` for this TF-IDF representation. `IsolationForest` often predicts almost everything as normal, especially the `default`, `more_trees`, and `strict` configs, which produce near-zero recall and F1.

The `loose` configs improve recall by allowing more anomaly flags, but they also increase FPR. For example, `sgd_one_class_svm:loose` has the best test F1 but also the highest FPR among the top unsupervised runs. The tradeoff is acceptable only if catching more toxic examples matters more than avoiding false alarms.

Overall, these results suggest unsupervised normal-only training is useful as a high-precision anomaly signal, but not enough for primary toxicity classification. The next step should be comparing this against supervised models and possibly using unsupervised scores as an additional feature or triage signal rather than as the final classifier.